# NB03: Evaluate All Checkpoints

**Chay: Internet ON (de download seq9), GPU ON**

Input: Checkpoints tu NB02 (Kaggle Dataset hoac local)

Output: `results_all.csv` + `results_summary.csv` + visualization

Metrics: mIoU, PA, mF1, Per-class IoU (11 class), Params(M), FLOPs(G), FPS

In [ ]:
# ============================================================
# CONFIG — chi can thay doi o day
# ============================================================
IMG_SIZE    = (768, 768)
BATCH_SIZE  = 8
NUM_WORKERS = 4
USE_AMP     = True

import os
IS_KAGGLE = os.path.exists('/kaggle')
from pathlib import Path

# >>> SUA TEN DATASET checkpoints cua ban tai day <<<
# Sau khi Save Version NB02 -> Add Data -> tim ten notebook do
CKPT_DATASET_NAME = 'forest-checkpoints'   # ten slug Kaggle dataset

CKPT_ROOT = Path(f'/kaggle/input/{CKPT_DATASET_NAME}') if IS_KAGGLE \
            else Path('outputs/checkpoints')
OUT_DIR   = Path('/kaggle/working/eval_results') if IS_KAGGLE \
            else Path('outputs/eval_results')
OUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_SEQ = 'seq9'   # seq9 phai duoc upload rieng tu Zenodo

print(f'CKPT_ROOT: {CKPT_ROOT}')
print(f'OUT_DIR:   {OUT_DIR}')
print(f'CKPT_ROOT exists: {CKPT_ROOT.exists()}')

In [ ]:
# ============================================================
# DOWNLOAD SEQ9 (Test Set) tu Zenodo
# ============================================================
# seq9 KHONG co trong Kaggle datasets (part1: seq1-4, part2: seq5-8)
# Luon download tu Zenodo — CAN INTERNET ON
#
# Source: Blaga & Nedevschi (2026), Scientific Data
# DOI:   https://doi.org/10.5281/zenodo.15511426
# ============================================================
import os, zipfile
from pathlib import Path

IS_KAGGLE  = os.path.exists('/kaggle')
SEQ9_DIR   = Path('/kaggle/working/data/seq9') if IS_KAGGLE else Path('data/forest_sunny/seq9')
ZENODO_URL = 'https://zenodo.org/records/15511426/files/seq9.zip'

if SEQ9_DIR.exists() and (SEQ9_DIR / 'color').exists():
    n = len(list((SEQ9_DIR / 'color').glob('*.png')))
    print(f'[OK] seq9 da co san: {SEQ9_DIR} ({n} images) — skip download')
else:
    import urllib.request
    zip_path = Path('/kaggle/working/seq9.zip') if IS_KAGGLE else Path('data/seq9.zip')
    zip_path.parent.mkdir(parents=True, exist_ok=True)

    print(f'[DOWNLOAD] Dang tai seq9 tu Zenodo...')
    print(f'  URL: {ZENODO_URL}')
    print(f'  Co the mat 5-15 phut tuy toc do mang.')

    def _progress(count, block_size, total_size):
        pct = count * block_size * 100 / total_size if total_size > 0 else 0
        mb  = count * block_size / 1024 / 1024
        print(f'\r  Progress: {pct:.1f}% ({mb:.0f} MB)', end='', flush=True)

    urllib.request.urlretrieve(ZENODO_URL, str(zip_path), reporthook=_progress)
    print(f'\n  Downloaded: {zip_path.stat().st_size / 1024**2:.0f} MB')

    # Giai nen
    print('  Extracting...')
    SEQ9_DIR.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(SEQ9_DIR.parent)
    zip_path.unlink(missing_ok=True)

    n = len(list((SEQ9_DIR / 'color').glob('*.png')))
    print(f'  [OK] seq9 ready: {SEQ9_DIR} ({n} images)')

# Verify
assert (SEQ9_DIR / 'color').exists(),  f'FAIL: {SEQ9_DIR}/color not found'
assert (SEQ9_DIR / 'labels').exists(), f'FAIL: {SEQ9_DIR}/labels not found'
n_color  = len(list((SEQ9_DIR / 'color').glob('*.png')))
n_labels = len(list((SEQ9_DIR / 'labels').glob('*.png')))
print(f'\nseq9 verified: {n_color} color, {n_labels} labels')


In [ ]:
# ============================================================
# SETUP
# ============================================================
!pip install -q segmentation-models-pytorch albumentations fvcore seaborn

import re, time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from PIL import Image
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

In [ ]:
# ============================================================
# DATASET CONSTANTS
# ============================================================
LABEL_COLORS = [
    (0,255,255),(0,127,0),(19,132,69),(0,53,65),(130,76,0),
    (152,251,152),(151,126,171),(250,150,0),(115,176,195),(123,123,123),(0,0,0)
]
CLASS_NAMES = [
    'Sky','Deciduous_trees','Coniferous_trees','Fallen_trees','Dirt_ground',
    'Ground_vegetation','Rocks','Building','Fence','Car','Empty'
]
NUM_CLASSES = len(CLASS_NAMES)
PALETTE     = np.array(LABEL_COLORS, dtype=np.uint8)

# Color -> class lookup table (O(1) per pixel)
COLOR_LUT = np.full((256, 256, 256), NUM_CLASSES - 1, dtype=np.int64)
for i, (r, g, b) in enumerate(LABEL_COLORS):
    COLOR_LUT[r, g, b] = i

def rgb_to_mask(rgb_np):
    return COLOR_LUT[rgb_np[:,:,0], rgb_np[:,:,1], rgb_np[:,:,2]]

def mask_to_rgb(mask):
    h, w = mask.shape
    out = np.zeros((h, w, 3), dtype=np.uint8)
    for i in range(NUM_CLASSES):
        out[mask == i] = PALETTE[i]
    return out

class ForestDataset(Dataset):
    def __init__(self, root, sequences, transform=None):
        self.transform = transform
        self.samples   = []
        root = Path(root)
        for seq in sequences:
            color_dir = root / seq / 'color'
            if not color_dir.exists():
                print(f'  [SKIP] {seq} — no color dir')
                continue
            for f in sorted(color_dir.glob('*.png')):
                lf = root / seq / 'labels' / f.name
                if lf.exists():
                    self.samples.append((f, lf))
        print(f'  Loaded: {len(self.samples)} samples from {sequences}')

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        img_path, lbl_path = self.samples[i]
        img  = np.array(Image.open(img_path).convert('RGB'))
        mask = rgb_to_mask(np.array(Image.open(lbl_path).convert('RGB')))
        if self.transform:
            t = self.transform(image=img, mask=mask.astype(np.int64))
            img, mask = t['image'], t['mask']
        return img.float(), mask.long(), str(img_path)

# Locate data root (support Kaggle symlink structure)
if IS_KAGGLE:
    wdata = Path('/kaggle/working/data')
    wdata.mkdir(exist_ok=True)
    for dirpath, _, _ in os.walk('/kaggle/input', followlinks=True):
        if os.path.basename(dirpath) == 'color':
            seq_dir  = os.path.dirname(dirpath)
            seq_name = os.path.basename(seq_dir)
            if re.match(r'seq\d+$', seq_name):
                link = wdata / seq_name
                if not link.exists():
                    os.symlink(seq_dir, link)
    DATA_ROOT = wdata
else:
    DATA_ROOT = Path('data/forest_sunny')

test_tf = A.Compose([
    A.Resize(*IMG_SIZE),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])
test_ds     = ForestDataset(DATA_ROOT, [TEST_SEQ], test_tf)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=(DEVICE=='cuda'))
print(f'Test set: {len(test_ds)} samples')

In [ ]:
# ============================================================
# MODEL REGISTRY & METRIC HELPERS
# ============================================================
MODELS = {
    'unet':          lambda enc: smp.Unet(encoder_name=enc, encoder_weights=None, in_channels=3, classes=NUM_CLASSES),
    'unetpp':        lambda enc: smp.UnetPlusPlus(encoder_name=enc, encoder_weights=None, in_channels=3, classes=NUM_CLASSES),
    'deeplabv3plus': lambda enc: smp.DeepLabV3Plus(encoder_name=enc, encoder_weights=None, in_channels=3, classes=NUM_CLASSES),
    'upernet':       lambda enc: smp.UPerNet(encoder_name=enc, encoder_weights=None, in_channels=3, classes=NUM_CLASSES),
    'segformer':     lambda enc: smp.Segformer(encoder_name=enc, encoder_weights=None, in_channels=3, classes=NUM_CLASSES),
}

def count_params(model):
    return sum(p.numel() for p in model.parameters()) / 1e6

def calc_flops(model):
    try:
        from fvcore.nn import FlopCountAnalysis
        dummy = torch.zeros(1, 3, *IMG_SIZE).to(next(model.parameters()).device)
        fa = FlopCountAnalysis(model, dummy)
        fa.unsupported_ops_warnings(False)
        fa.uncalled_modules_warnings(False)
        return fa.total() / 1e9
    except Exception as e:
        print(f'  [FLOPs WARN] {e}')
        return -1.0

def calc_fps(model, n_runs=50):
    if DEVICE != 'cuda': return -1.0
    model.eval()
    dummy = torch.zeros(1, 3, *IMG_SIZE).to(DEVICE)
    with torch.no_grad():
        for _ in range(10): model(dummy)        # warmup
    torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        for _ in range(n_runs): model(dummy)
    torch.cuda.synchronize()
    return n_runs / (time.time() - t0)

def compute_metrics(cm):
    """mIoU, PA, mF1 + per-class from NxN confusion matrix."""
    inter = np.diag(cm)
    union = cm.sum(1) + cm.sum(0) - inter
    iou   = np.where(union > 0, inter / union, np.nan)
    miou  = np.nanmean(iou)
    pa    = inter.sum() / (cm.sum() + 1e-9)
    prec  = np.where(cm.sum(0) > 0, inter / cm.sum(0), np.nan)
    rec   = np.where(cm.sum(1) > 0, inter / cm.sum(1), np.nan)
    f1    = np.where((prec + rec) > 0, 2*prec*rec/(prec+rec), np.nan)
    mf1   = np.nanmean(f1)
    return {'mIoU': miou, 'PA': pa, 'mF1': mf1,
            'iou_per_class': iou, 'f1_per_class': f1}

def evaluate_checkpoint(ckpt_path, model_name, encoder):
    model = MODELS[model_name](encoder).to(DEVICE)
    # weights_only=False: support checkpoints that saved extra metadata
    ckpt  = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    state = ckpt.get('model_state_dict', ckpt)   # support bare state_dict too
    model.load_state_dict(state)
    model.eval()

    params = count_params(model)
    flops  = calc_flops(model)
    fps    = calc_fps(model)

    cm           = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    saved_samples = []

    with torch.no_grad():
        for imgs, masks, _ in tqdm(test_loader, desc=f'{model_name}/{encoder[:10]}', leave=False):
            imgs  = imgs.to(DEVICE)
            with autocast('cuda', enabled=USE_AMP and DEVICE == 'cuda'):
                out = model(imgs)
            preds = out.argmax(1).cpu().numpy()    # (B, H, W)
            gts   = masks.numpy()                  # (B, H, W)

            # --- Vectorized CM update (much faster than pixel loop) ---
            for pred, gt in zip(preds, gts):
                valid = (gt >= 0) & (gt < NUM_CLASSES)
                np.add.at(cm, (gt[valid], pred[valid]), 1)

            # Save first 8 samples for qualitative viz
            if len(saved_samples) < 8:
                for i in range(min(len(imgs), 8 - len(saved_samples))):
                    img_np = imgs[i].cpu().numpy().transpose(1,2,0)
                    mean_n = np.array([0.485, 0.456, 0.406])
                    std_n  = np.array([0.229, 0.224, 0.225])
                    img_np = np.clip(img_np * std_n + mean_n, 0, 1)
                    saved_samples.append((img_np, preds[i], gts[i]))

    metrics = compute_metrics(cm)
    metrics.update({'params_M': params, 'flops_G': flops, 'fps': fps,
                    'confusion_matrix': cm, 'saved_samples': saved_samples})
    return metrics

print('Helpers ready.')

In [ ]:
# ============================================================
# SCAN CHECKPOINTS
# ============================================================
# Folder naming convention: <model>_<encoder>_seed<N>
# Example: segformer_mit_b5_seed42  /  unet_resnet34_seed123
# Inside folder: best checkpoint .pth file (name may contain miou_XX.XXXX)

def find_best_ckpt(folder):
    pths = sorted(Path(folder).glob('*.pth'))
    if not pths: return None
    def get_miou(p):
        m = re.search(r'miou_([0-9.]+)', p.name)
        return float(m.group(1)) if m else 0.0
    return max(pths, key=get_miou)

EXPERIMENTS = []
if CKPT_ROOT.exists():
    for folder in sorted(CKPT_ROOT.iterdir()):
        if not folder.is_dir(): continue
        m = re.match(r'(.+)_seed(\d+)$', folder.name)
        if not m:
            print(f'[SKIP] Cannot parse folder name: {folder.name}')
            continue
        config_name = m.group(1)              # e.g. segformer_mit_b5
        seed        = int(m.group(2))
        parts       = config_name.split('_', 1)
        model_name  = parts[0]                # e.g. segformer
        encoder     = parts[1] if len(parts) > 1 else ''

        if model_name not in MODELS:
            print(f'[SKIP] Unknown model: {model_name} in {folder.name}')
            continue

        best_ckpt = find_best_ckpt(folder)
        if best_ckpt:
            EXPERIMENTS.append({
                'config': config_name, 'model': model_name,
                'encoder': encoder, 'seed': seed, 'ckpt': best_ckpt
            })
            print(f'  [FOUND] {folder.name:45s} -> {best_ckpt.name}')
        else:
            print(f'  [EMPTY] {folder.name} — no .pth files')
else:
    print(f'[ERROR] CKPT_ROOT not found: {CKPT_ROOT}')
    print('  -> Kiem tra CKPT_DATASET_NAME trong cell CONFIG')

print(f'\nTotal: {len(EXPERIMENTS)} checkpoints to evaluate')

In [ ]:
# ============================================================
# EVALUATE ALL CHECKPOINTS
# ============================================================
all_results = []

for i, exp in enumerate(EXPERIMENTS, 1):
    print(f'\n[{i}/{len(EXPERIMENTS)}] {exp["config"]} seed={exp["seed"]}')
    try:
        metrics = evaluate_checkpoint(exp['ckpt'], exp['model'], exp['encoder'])

        row = {
            'config':   exp['config'],
            'model':    exp['model'],
            'encoder':  exp['encoder'],
            'seed':     exp['seed'],
            'ckpt':     str(exp['ckpt']),
            'mIoU':     round(float(metrics['mIoU']) * 100, 2),
            'PA':       round(float(metrics['PA'])   * 100, 2),
            'mF1':      round(float(metrics['mF1'])  * 100, 2),
            'params_M': round(float(metrics['params_M']), 2),
            'flops_G':  round(float(metrics['flops_G']),  2),
            'fps':      round(float(metrics['fps']),      1),
        }
        for j, cls in enumerate(CLASS_NAMES):
            v = metrics['iou_per_class'][j]
            row[f'iou_{cls}'] = round(float(v) * 100, 2) if not np.isnan(v) else -1.0

        all_results.append(row)
        print(f'  mIoU={row["mIoU"]}% | PA={row["PA"]}% | '
              f'mF1={row["mF1"]}% | Params={row["params_M"]}M | FPS={row["fps"]}')

        # Save confusion matrix + qualitative samples
        np.save(OUT_DIR / f'{exp["config"]}_seed{exp["seed"]}_cm.npy',
                metrics['confusion_matrix'])
        np.save(OUT_DIR / f'{exp["config"]}_seed{exp["seed"]}_samples.npy',
                np.array(metrics['saved_samples'], dtype=object), allow_pickle=True)

        # Auto-save after each checkpoint (avoid losing work)
        pd.DataFrame(all_results).to_csv(OUT_DIR / 'results_all.csv', index=False)

    except Exception as e:
        print(f'  [ERROR] {e}')
        import traceback; traceback.print_exc()

df = pd.DataFrame(all_results)
df.to_csv(OUT_DIR / 'results_all.csv', index=False)
print(f'\nFinal save: {len(all_results)}/{len(EXPERIMENTS)} evaluated')

In [ ]:
# ============================================================
# SUMMARY TABLE — mean +/- std per config
# ============================================================
df = pd.read_csv(OUT_DIR / 'results_all.csv')

agg_cols = ['mIoU', 'PA', 'mF1', 'params_M', 'flops_G', 'fps']
mean_df  = df.groupby('config')[agg_cols].mean().round(2)
std_df   = df.groupby('config')[agg_cols].std().round(2).fillna(0)

# Build readable summary (keep mIoU as float for sort, then format)
summary = mean_df[agg_cols].copy()
summary['mIoU_std'] = std_df['mIoU']
summary['PA_std']   = std_df['PA']
summary['mF1_std']  = std_df['mF1']
summary = summary.sort_values('mIoU', ascending=False)

# Display formatted
display_df = pd.DataFrame(index=summary.index)
display_df['mIoU']     = summary['mIoU'].astype(str) + ' +/- ' + summary['mIoU_std'].astype(str)
display_df['PA']       = summary['PA'].astype(str)   + ' +/- ' + summary['PA_std'].astype(str)
display_df['mF1']      = summary['mF1'].astype(str)  + ' +/- ' + summary['mF1_std'].astype(str)
display_df['Params(M)']= summary['params_M']
display_df['FLOPs(G)'] = summary['flops_G']
display_df['FPS']      = summary['fps']

print('=== Results Summary (mean +/- std, 3 seeds) ===')
print(display_df.to_string())

# Save numeric version for NB04
summary.to_csv(OUT_DIR / 'results_summary.csv')

print(f'\nBest: {summary.index[0]} | mIoU: {summary["mIoU"].iloc[0]:.2f}%')

In [ ]:
# ============================================================
# VIZ 1: Qualitative Predictions (best model)
# ============================================================
best_config = summary.index[0]
best_seed   = df[df['config'] == best_config].sort_values('mIoU', ascending=False)['seed'].iloc[0]
viz_path    = OUT_DIR / f'{best_config}_seed{best_seed}_samples.npy'

if viz_path.exists():
    samples = np.load(viz_path, allow_pickle=True)
    n = min(6, len(samples))
    fig, axes = plt.subplots(n, 3, figsize=(15, n * 4))
    if n == 1: axes = axes[np.newaxis]

    for i in range(n):
        img, pred, gt = samples[i]
        axes[i, 0].imshow(img)
        axes[i, 0].set_title('Input', fontsize=11); axes[i, 0].axis('off')
        axes[i, 1].imshow(mask_to_rgb(gt))
        axes[i, 1].set_title('Ground Truth', fontsize=11); axes[i, 1].axis('off')
        axes[i, 2].imshow(mask_to_rgb(pred))
        axes[i, 2].set_title(f'Prediction', fontsize=11); axes[i, 2].axis('off')

    patches = [mpatches.Patch(color=np.array(LABEL_COLORS[i])/255, label=CLASS_NAMES[i])
               for i in range(NUM_CLASSES)]
    fig.legend(handles=patches, loc='lower center', ncol=6, bbox_to_anchor=(0.5, -0.01),
               fontsize=9, frameon=True)
    plt.suptitle(f'Qualitative Results — {best_config} (seed={best_seed})', fontsize=13, y=1.01)
    plt.tight_layout()
    out_path = OUT_DIR / 'figure_qualitative.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_path}')
else:
    print(f'[SKIP] {viz_path} not found')

In [ ]:
# ============================================================
# VIZ 2: Confusion Matrix (best model)
# ============================================================
cm_path = OUT_DIR / f'{best_config}_seed{best_seed}_cm.npy'

if cm_path.exists():
    cm      = np.load(cm_path)
    cm_norm = cm.astype(float) / (cm.sum(1, keepdims=True) + 1e-9)

    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, linewidths=0.5, vmin=0, vmax=1)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Ground Truth', fontsize=12)
    ax.set_title(f'Normalized Confusion Matrix — {best_config}', fontsize=13)
    plt.xticks(rotation=40, ha='right', fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()
    out_path = OUT_DIR / 'figure_confusion_matrix.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_path}')
else:
    print(f'[SKIP] {cm_path} not found')

In [ ]:
print('=' * 60)
print('NB03 XONG!')
print('=' * 60)
outs = list(OUT_DIR.iterdir())
csvs = [f for f in outs if f.suffix == '.csv']
pngs = [f for f in outs if f.suffix == '.png']
print(f'  CSVs:   {[f.name for f in csvs]}')
print(f'  Figs:   {[f.name for f in pngs]}')
print(f'  .npy:   {len([f for f in outs if f.suffix == ".npy"])} files')
print()
print('BUOC TIEP THEO:')
print('  1. Save Version (de xuat CSV ra ngoai)')
print('  2. Add output nay vao NB04 de tao paper figures + LaTeX tables')